# Phase 1: Environment Setup and Dataset Preprocessing

In this notebook, we prepare the Colab/Kaggle environment, import the `IPPy` library, and process the *Egyptian Medical Retinal Images* dataset. We save the processed tensors permanently on Google Drive to avoid recomputing them in future sessions.

## 1.1 Google Drive Connection and Environment Setup

Run the following cell to mount your Google Drive. We will create a `CI_Project` folder to host the dataset and the trained model weights.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os

if IN_COLAB:
    drive.mount('/content/drive')
    # --- CONFIGURATION ---
    # Set your preferred base directory on Google Drive
    BASE_DRIVE_FOLDER = 'CI_Project'
    # ---------------------
    DRIVE_PATH = os.path.join('/content/drive/MyDrive', BASE_DRIVE_FOLDER)
else:
    # Fallback for local testing if needed
    DRIVE_PATH = './CI_Project'

DATASET_PATH = os.path.join(DRIVE_PATH, 'dataset')
WEIGHTS_PATH = os.path.join(DRIVE_PATH, 'weights')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(WEIGHTS_PATH, exist_ok=True)

print(f"Data will be permanently saved in: {DRIVE_PATH}")

## 1.2 Cloning IPPy and Installing Dependencies
We clone the custom fork of the IPPy repository which contains the solvers and operators, and install required libraries such as `PyWavelets`.

In [ ]:
import subprocess
import sys

# Clone or update the IPPy repository
if not os.path.exists('IPPy'):
    !git clone https://github.com/alessandrocampedelli/IPPy.git
else:
    %cd IPPy
    !git pull
    %cd ..

# Install extra dependencies
!pip install -q PyWavelets astra-toolbox kagglehub

# Add IPPy to the Python path
sys.path.append(os.path.abspath('IPPy'))

# Verify import
from IPPy.operators import DownScaling
from IPPy.utilities._utilities import gaussian_noise
print("IPPy library imported successfully!")

## 1.3 Dataset Download and Preprocessing

We process the ~4000 images, resizing them to `256x256` and splitting them into Train, Validation, and Test sets. These splits are saved on Drive.

*Note: Replace the download link/command with the actual Kaggle dataset command or direct URL if the zip is not already on your Drive.*

In [ ]:
import glob
from PIL import Image
import torch
import torchvision.transforms as T
from torch.utils.data import random_split
from tqdm.auto import tqdm
import shutil

import kagglehub

print('Downloading dataset via kagglehub...')
# This downloads the dataset into a cache folder (or retrieves it if already downloaded)
RAW_IMAGES_DIR = kagglehub.dataset_download("mohamedabdalkader/egyptian-medical-retinal-images")
print('Path to dataset files:', RAW_IMAGES_DIR)

class RetinalDatasetBuilder:
    def __init__(self, source_dir, target_dir, target_size=(256, 256)):
        self.source_dir = source_dir
        self.target_dir = target_dir
        self.transform = T.Compose([
            T.Grayscale(num_output_channels=1),
            T.Resize(target_size),
            T.ToTensor() # Maps to [0, 1]
        ])
        
    def process_and_save(self):
        if not os.path.exists(self.source_dir):
            print(f"Directory {self.source_dir} not found. Skipping preprocessing.")
            print("Please upload or download the dataset first.")
            return
            
        # Recursively find all images (jpg, png, jpeg)
        img_paths = glob.glob(os.path.join(self.source_dir, '**', '*.*'), recursive=True)
        img_paths = [p for p in img_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        if not img_paths:
            print("No images found in the specified directory.")
            return
            
        print(f"Found {len(img_paths)} images. Starting preprocessing...")
        
        # Split: 80% train, 10% val, 10% test
        torch.manual_seed(42)
        train_len = int(0.8 * len(img_paths))
        val_len = int(0.1 * len(img_paths))
        test_len = len(img_paths) - train_len - val_len
        
        splits = random_split(img_paths, [train_len, val_len, test_len])
        split_names = ['train', 'val', 'test']
        
        for split_data, name in zip(splits, split_names):
            split_dir = os.path.join(self.target_dir, name)
            os.makedirs(split_dir, exist_ok=True)
            
            for path in tqdm(split_data, desc=f"Processing {name} set"):
                try:
                    img = Image.open(path)
                    tensor_img = self.transform(img) # Shape: (1, 256, 256)
                    
                    filename = os.path.basename(path)
                    filename_pt = os.path.splitext(filename)[0] + '.pt'
                    
                    # Save the tensor for ultra-fast loading in future notebooks
                    torch.save(tensor_img, os.path.join(split_dir, filename_pt))
                except Exception as e:
                    print(f"Error processing {path}: {e}")
                
        print(f"Dataset successfully preprocessed and saved to {self.target_dir}!")

# Execute builder only if the Drive folder is empty
train_dir = os.path.join(DATASET_PATH, 'train')
if not os.path.exists(train_dir) or len(os.listdir(train_dir)) == 0:
    builder = RetinalDatasetBuilder(RAW_IMAGES_DIR, DATASET_PATH)
    builder.process_and_save()
else:
    print(f"Dataset already exists in {DATASET_PATH}. No preprocessing required!")

## 1.4 Forward Model Validation
Now we verify that the `DownScaling` operator (with scale factors $s=2$ and $s=4$) and the Gaussian noise function ($\sigma=0.005$ and $\sigma=0.01$) work correctly on a test image loaded directly from Drive.

In [ ]:
import matplotlib.pyplot as plt
import random

# Load a random test image
test_dir = os.path.join(DATASET_PATH, 'test')
if os.path.exists(test_dir):
    test_files = glob.glob(os.path.join(test_dir, '*.pt'))
    if test_files:
        test_file = random.choice(test_files)
        x_true = torch.load(test_file).unsqueeze(0) # Shape: (1, 1, 256, 256)
        
        # Initialize operators
        H2 = DownScaling(img_shape=(256, 256), downscale_factor=2, mode="avg")
        H4 = DownScaling(img_shape=(256, 256), downscale_factor=4, mode="avg")
        
        # Create degraded observations (Forward Model: y = Hx + e)
        # Factor 2, Noise 0.005
        y2_clean = H2(x_true)
        y2 = y2_clean + gaussian_noise(y2_clean, noise_level=0.005)
        
        # Factor 4, Noise 0.01
        y4_clean = H4(x_true)
        y4 = y4_clean + gaussian_noise(y4_clean, noise_level=0.01)
        
        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(x_true[0, 0].numpy(), cmap='gray')
        axes[0].set_title('Original (256x256)')
        axes[0].axis('off')
        
        axes[1].imshow(y2[0, 0].numpy(), cmap='gray')
        axes[1].set_title('Degraded s=2, σ=0.005 (128x128)')
        axes[1].axis('off')
        
        axes[2].imshow(y4[0, 0].numpy(), cmap='gray')
        axes[2].set_title('Degraded s=4, σ=0.01 (64x64)')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No test images found. Please run preprocessing first.")
else:
    print(f"Test directory not found at {test_dir}")